In [2]:
import numpy as np

from collections import deque

import matplotlib.pyplot as plt

%matplotlib inline

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
# Gym
import gym

# Hugging Face Hub
from huggingface_hub import (
    notebook_login,
)  # To log to our Hugging Face account to be able to upload models to the Hub.
import imageio


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [6]:
env_id = "Pixelcopter-PLE-v0"
env = gym.make(env_id)
eval_env = gym.make(env_id)
s_size = env.observation_space.shape[0]
a_size = env.action_space.n
print(f"State space size: {s_size}")
print(f"Action space size: {a_size}")

NameNotFound: Environment Pixelcopter-PLE doesn't exist. 

In [80]:
# Preprocessing function (adjust based on your model's input requirements)
def preprocess_state(state):
    """
    Preprocess the game state for the neural network
    This should match the preprocessing used during training
    """
    if isinstance(state, np.ndarray) and len(state.shape) == 3:
        # If using image input
        state = np.transpose(state, (2, 0, 1))  # Channel first
        state = state / 255.0  # Normalize pixels
        return state.flatten()  # or keep as image depending on model
    else:
        # If using game state features
        return np.array(list(state.values()))


In [81]:
env = create_pixelcopter_env()
eval_env = create_pixelcopter_env()
env.init()
s = env.getGameState()
s

{'player_y': 24.0,
 'player_vel': 0,
 'player_dist_to_ceil': 7.0,
 'player_dist_to_floor': 17.0,
 'next_gate_dist_to_player': 48,
 'next_gate_block_top': 22,
 'next_gate_block_bottom': 31}

In [82]:
acts = env.getActionSet()
acts

[119, None]

In [83]:
env.init()
for _ in range(10):
    s = env.getGameState()
    print(str(s) + "\n")
    action = 1
    reward = env.act(action)
    print(f"Reward: {reward} \n")

{'player_y': 24.0, 'player_vel': 0, 'player_dist_to_ceil': 7.0, 'player_dist_to_floor': 17.0, 'next_gate_dist_to_player': 49, 'next_gate_block_top': 31, 'next_gate_block_bottom': 40}

Reward: 0.0 

{'player_y': 24.057024, 'player_vel': 0.05702400000000001, 'player_dist_to_ceil': 7.057023999999998, 'player_dist_to_floor': 16.942976, 'next_gate_dist_to_player': 48.36, 'next_gate_block_top': 31, 'next_gate_block_bottom': 40}

Reward: 0.0 

{'player_y': 24.170501759999997, 'player_vel': 0.11347776000000002, 'player_dist_to_ceil': 7.170501759999997, 'player_dist_to_floor': 16.829498240000003, 'next_gate_dist_to_player': 47.72, 'next_gate_block_top': 31, 'next_gate_block_bottom': 40}

Reward: 0.0 

{'player_y': 24.339868742399997, 'player_vel': 0.16936698240000003, 'player_dist_to_ceil': 7.339868742399997, 'player_dist_to_floor': 16.660131257600003, 'next_gate_dist_to_player': 47.08, 'next_gate_block_top': 31, 'next_gate_block_bottom': 40}

Reward: 0.0 

{'player_y': 24.564566054975998, 'pla

In [84]:
s = preprocess_state(s)
s

array([26.49883445,  0.49315925, 10.49883445, 13.50116555, 43.24      ,
       31.        , 40.        ])

In [85]:
s_size = s.shape[0]
a_size = 2


In [86]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


cpu


In [87]:
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, h_size * 2)
        self.fc3 = nn.Linear(h_size * 2, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return F.softmax(x, dim=1)

    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)


In [88]:
pixelcopter_hyperparameters = {
    "h_size": 64,
    "n_training_episodes": 50000,
    "n_evaluation_episodes": 10,
    "max_t": 10000,
    "gamma": 0.99,
    "lr": 1e-2,
    "env_id": "pixelcopter",
    "state_space": s_size,
    "action_space": a_size,
}
# Create policy and place it to the device
# torch.manual_seed(50)
pixelcopter_policy = Policy(
    pixelcopter_hyperparameters["state_space"],
    pixelcopter_hyperparameters["action_space"],
    pixelcopter_hyperparameters["h_size"],
).to(device)
pixelcopter_optimizer = optim.Adam(
    pixelcopter_policy.parameters(), lr=pixelcopter_hyperparameters["lr"]
)

In [ ]:
def reinforce(policy, optimizer, n_training_episodes, max_t, gamma, print_every):
    # Help us to calculate the score during the training
    scores_deque = deque(maxlen=max_t)
    scores = []
    # Line 3 of pseudocode
    for i_episode in range(1, n_training_episodes + 1):
        saved_log_probs = []
        rewards = []
        env.reset_game()

        # Line 4 of pseudocode
        for t in range(max_t):
            state = env.getGameState()
            state = preprocess_state(state)  # Apply your preprocessing
            action, log_prob = policy.act(state)
            saved_log_probs.append(log_prob)
            reward = env.act(action)
            rewards.append(reward)
            if env.game_over():
                break
            
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        # Line 6 of pseudocode: calculate the return
        returns = deque(maxlen=max_t)
        n_steps = len(rewards)
        # Compute the discounted returns at each timestep,
        for t in range(n_steps)[::-1]:
            disc_return_t = returns[0] if len(returns) > 0 else 0
            returns.appendleft(gamma * disc_return_t + rewards[t])

        ## standardization of the returns is employed to make training more stable
        eps = np.finfo(np.float32).eps.item()
        ## eps is the smallest representable float, which is
        # added to the standard deviation of the returns to avoid numerical instabilities
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        # Line 7:
        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.cat(policy_loss).sum()

        # Line 8: PyTorch prefers gradient descent
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if i_episode % print_every == 0:
            print(
                "Episode {}\tAverage Score: {:.2f}".format(
                    i_episode, np.mean(scores_deque)
                )
            )

    return scores


In [91]:
scores = reinforce(
    pixelcopter_policy,
    pixelcopter_optimizer,
    pixelcopter_hyperparameters["n_training_episodes"],
    pixelcopter_hyperparameters["max_t"],
    pixelcopter_hyperparameters["gamma"],
    1000,
)


Episode 1000	Average Score: -2.65
Episode 2000	Average Score: -2.61
Episode 3000	Average Score: -2.65
Episode 4000	Average Score: -2.65
Episode 5000	Average Score: -2.67
Episode 6000	Average Score: -2.63
Episode 7000	Average Score: -2.65
Episode 8000	Average Score: -2.69
Episode 9000	Average Score: -2.61
Episode 10000	Average Score: -2.64
Episode 11000	Average Score: -2.64
Episode 12000	Average Score: -2.66
Episode 13000	Average Score: -2.63
Episode 14000	Average Score: -2.59
Episode 15000	Average Score: -2.63
Episode 16000	Average Score: -2.65
Episode 17000	Average Score: -2.68
Episode 18000	Average Score: -2.62
Episode 19000	Average Score: -2.65
Episode 20000	Average Score: -2.61
Episode 21000	Average Score: -2.60
Episode 22000	Average Score: -2.70
Episode 23000	Average Score: -2.69
Episode 24000	Average Score: -2.65
Episode 25000	Average Score: -2.56
Episode 26000	Average Score: -2.56
Episode 27000	Average Score: -2.61
Episode 28000	Average Score: -2.65
Episode 29000	Average Score: 